# Receipt Scanner - PaddleOCR Prototype

This notebook demonstrates extracting expense data from receipt images using **PaddleOCR**.

**Fields extracted:**
- `description` - Store/vendor name
- `amount` - Total amount
- `category` - food, transport, accommodation, entertainment, utilities, shopping, health, education, other

## 1. Install Dependencies

In [ ]:
!pip install paddlepaddle paddleocr Pillow httpx numpy

## 2. Initialize PaddleOCR

In [ ]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(use_angle_cls=True, lang='en', show_log=False)
print('PaddleOCR initialized successfully!')

## 3. Download a Test Receipt Image

Using a sample receipt image URL for testing. Replace with your S3 URL.

In [ ]:
import httpx
import io
import numpy as np
from PIL import Image

# Sample receipt image for testing
# Replace this URL with your S3 presigned URL
IMAGE_URL = "https://ocr.space/Content/Images/receipt-ocr-original.webp"

response = httpx.get(IMAGE_URL, follow_redirects=True, timeout=30.0)
response.raise_for_status()
image_bytes = response.content

img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
print(f'Image downloaded: {img.size[0]}x{img.size[1]} pixels')
img

## 4. Run PaddleOCR - Extract Raw Text

In [ ]:
img_array = np.array(img)
result = ocr.ocr(img_array, cls=True)

# Extract all text lines
lines = []
if result and result[0]:
    for line in result[0]:
        text = line[1][0]
        confidence = line[1][1]
        lines.append(text)
        print(f'[{confidence:.2f}] {text}')

ocr_text = '\n'.join(lines)
print('\n--- Full OCR Text ---')
print(ocr_text)

## 5. Extract Amount (Total)

In [ ]:
import re

def extract_amount(ocr_text: str):
    """Extract the total/grand total amount from OCR text."""
    text_lower = ocr_text.lower()

    total_patterns = [
        r"grand\s*total\s*[:\-]?\s*[\$\₹\€\£]?\s*([\d,]+\.?\d*)",
        r"total\s*amount\s*[:\-]?\s*[\$\₹\€\£]?\s*([\d,]+\.?\d*)",
        r"net\s*total\s*[:\-]?\s*[\$\₹\€\£]?\s*([\d,]+\.?\d*)",
        r"total\s*[:\-]?\s*[\$\₹\€\£]?\s*([\d,]+\.?\d*)",
        r"amount\s*(?:due|paid|payable)?\s*[:\-]?\s*[\$\₹\€\£]?\s*([\d,]+\.?\d*)",
        r"[\$\₹\€\£]\s*([\d,]+\.?\d*)",
    ]

    for pattern in total_patterns:
        matches = re.findall(pattern, text_lower)
        if matches:
            amount_str = matches[-1].replace(',', '')
            try:
                return float(amount_str)
            except ValueError:
                continue

    # Fallback: largest number with decimals
    all_numbers = re.findall(r'([\d,]+\.\d{2})', ocr_text)
    if all_numbers:
        amounts = [float(n.replace(',', '')) for n in all_numbers]
        return max(amounts)

    return None


amount = extract_amount(ocr_text)
print(f'Extracted Amount: {amount}')

## 6. Extract Description (Store Name)

In [ ]:
def extract_description(ocr_text: str) -> str:
    """Extract store/vendor name from the receipt (usually the first lines)."""
    lines = [l.strip() for l in ocr_text.split('\n') if l.strip()]
    if not lines:
        return 'Receipt expense'

    for line in lines[:5]:
        cleaned = re.sub(r'[^a-zA-Z0-9\s&\'-]', '', line).strip()
        if len(cleaned) >= 3 and not re.match(r'^\d+$', cleaned):
            return cleaned

    return lines[0][:100] if lines else 'Receipt expense'


description = extract_description(ocr_text)
print(f'Extracted Description: {description}')

## 7. Classify Category

In [ ]:
CATEGORY_KEYWORDS = {
    'food': ['restaurant', 'cafe', 'pizza', 'burger', 'coffee', 'tea', 'bakery',
             'dominos', 'mcdonald', 'starbucks', 'subway', 'kfc', 'zomato',
             'swiggy', 'food', 'dine', 'eat', 'meal', 'lunch', 'dinner',
             'breakfast', 'snack', 'biryani', 'chicken', 'paneer', 'noodle',
             'rice', 'bread', 'juice', 'milk', 'grocery', 'supermarket',
             'bigbasket', 'blinkit', 'zepto', 'dmart', 'reliance fresh'],
    'transport': ['uber', 'ola', 'lyft', 'taxi', 'cab', 'fuel', 'petrol', 'diesel',
                  'gas station', 'parking', 'toll', 'metro', 'bus', 'train', 'flight',
                  'airline', 'airport', 'rapido', 'auto', 'rickshaw', 'indigo',
                  'irctc', 'redbus'],
    'accommodation': ['hotel', 'motel', 'inn', 'resort', 'airbnb', 'oyo', 'hostel',
                      'lodge', 'stay', 'room', 'check-in', 'checkout', 'booking',
                      'trivago', 'makemytrip'],
    'entertainment': ['movie', 'cinema', 'theatre', 'theater', 'concert', 'show',
                      'netflix', 'spotify', 'game', 'amusement', 'park', 'pvr', 'inox',
                      'bookmyshow', 'event', 'ticket'],
    'utilities': ['electric', 'electricity', 'water', 'gas', 'internet', 'wifi',
                  'broadband', 'phone', 'mobile', 'recharge', 'bill', 'jio',
                  'airtel', 'vodafone', 'bsnl'],
    'shopping': ['amazon', 'flipkart', 'myntra', 'ajio', 'mall', 'store', 'shop',
                 'retail', 'clothes', 'fashion', 'electronics', 'appliance',
                 'furniture', 'ikea', 'decathlon', 'nike', 'adidas', 'puma',
                 'meesho', 'nykaa'],
    'health': ['hospital', 'clinic', 'doctor', 'pharmacy', 'medical', 'medicine',
               'lab', 'diagnostic', 'apollo', 'fortis', 'medplus', 'netmeds',
               'pharmeasy', '1mg', 'health', 'dental', 'eye', 'optical'],
    'education': ['school', 'college', 'university', 'course', 'tuition', 'book',
                  'stationery', 'udemy', 'coursera', 'unacademy', 'byjus',
                  'library', 'exam', 'fee'],
}


def classify_category(ocr_text: str) -> str:
    text_lower = ocr_text.lower()
    scores = {}
    for category, keywords in CATEGORY_KEYWORDS.items():
        score = sum(1 for kw in keywords if kw in text_lower)
        if score > 0:
            scores[category] = score
    if scores:
        return max(scores, key=scores.get)
    return 'other'


category = classify_category(ocr_text)
print(f'Classified Category: {category}')

## 8. Final Result - Auto-fill Data

In [ ]:
expense_data = {
    'description': description,
    'amount': amount,
    'category': category,
}

print('=== Expense Auto-fill Data ===')
print(f"Description : {expense_data['description']}")
print(f"Amount      : {expense_data['amount']}")
print(f"Category    : {expense_data['category']}")
print()
print('Note: paid_by should be set to the user who submits the receipt (from auth/session)')

## 9. Test with FastAPI Endpoint

Start the server with:
```bash
cd receipt-scanner
uvicorn app.main:app --reload --port 8000
```

Then test from here:

In [ ]:
import httpx

API_URL = 'http://localhost:8000/api/scan-receipt'

# Replace with your S3 URL
payload = {
    'image_url': 'https://ocr.space/Content/Images/receipt-ocr-original.webp'
}

response = httpx.post(API_URL, json=payload, timeout=60.0)
result = response.json()

print('API Response:')
print(f"  Success     : {result['success']}")
if result['success']:
    data = result['data']
    print(f"  Description : {data['description']}")
    print(f"  Amount      : {data['amount']}")
    print(f"  Category    : {data['category']}")
    print(f"\nRaw OCR Text:\n{result['raw_ocr_text']}")
else:
    print(f"  Error: {result['error']}")